# IterativeFib (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/IterativeFib_java.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `IterativeFib`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/IterativeFib_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/IterativeFib_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono campi `static` della classe, e la traccia si attiva con `trace = true`.

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Fibonacci iterativo: IterativeFib

**Input:** un numero naturale $n$. **Output:** $F(n)$, calcolato da $F(0)$ e $F(1)$ in su tenendo solo gli ultimi due valori. $S(n) = 5n + 1 = \Theta(n)$ passi. Invariante: prima di ogni valutazione della condizione, $prev = F(i-2)$ e $curr = F(i-1)$.

Pseudocodice: sezione 14.5.2 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [2]:
%%writefile IterativeFib.java
public class IterativeFib
{
static long[] cnt = new long[20];   // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
static long calls = 0;              // numero di chiamate = nodi dell'albero delle chiamate
static int maxDepth = 0;            // profondita' della ricorsione = altezza massima dello stack
static long mults = 0;              // moltiplicazioni (FastPower)
static long moves = 0;              // mosse (Hanoi)
static int ev = 0;                  // contatore degli eventi della traccia
static java.util.ArrayList<Integer> stack = new java.util.ArrayList<>();   // lo stack delle chiamate: gli argomenti delle chiamate attive
static long[] seen = new long[64];  // seen[x] = numero di chiamate con argomento x
static boolean trace = false;       // true: stampa la traccia dell'esecuzione

static void reset()
{
    java.util.Arrays.fill(cnt, 0);
    java.util.Arrays.fill(seen, 0);
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0;
    stack.clear();
}

static long total()
{
    long s = 0;
    for (long c : cnt) s += c;
    return s;
}

static void printSteps(int... lines)
{
    System.out.println("Riga | Esecuzioni");
    for (int r : lines) System.out.println(String.format("%4d | %10d", r, cnt[r]));
    System.out.println("Passi totali S = " + total());
}

static void push(int x)             // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stack.add(x);
    if (stack.size() > maxDepth) maxDepth = stack.size();
    if (x >= 0 && x < 64) seen[x]++;
}

static void pop()                   // la chiamata termina: il suo frame esce dallo stack
{
    stack.remove(stack.size() - 1);
}

static void event(String text)      // riga numerata della traccia, con lo stack
{
    ev++;
    System.out.println(String.format("%2d. %s   stack: %s", ev, text, stack));
}

static void say(String call, String action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    System.out.println(String.format("%-28s| %s", "  ".repeat(stack.size() - 1) + call, action));
}

    // IterativeFib(n)
    static long iterativeFib(int n, boolean trace)
    {
        cnt[2]++;
        if (n <= 1)
        {
            cnt[3]++;
            return n;
        }
        long prev = 0; cnt[5]++;
        long curr = 1; cnt[6]++;
        int i = 2; cnt[7]++;
        if (trace)                                 // traccia: riga di inizializzazione
            System.out.println(String.format("%-10s | %-8s | %-18s | %4d | %4d", "inizio", "2", "-", prev, curr));
        cnt[8]++;
        while (i <= n)
        {
            long next = prev + curr; cnt[9]++;
            prev = curr; cnt[10]++;
            curr = next; cnt[11]++;
            i = i + 1; cnt[12]++;
            cnt[8]++;
            if (trace)                             // traccia: i - 2 e' il numero dell'iterazione
                System.out.println(String.format("%-10d | %-8s | %-18s | %4d | %4d", i - 2, String.format("%d -> %d", i - 1, i), String.format("%d + %d = %d", next - prev, prev, next), prev, curr));
        }
        cnt[14]++;
        return curr;
    }

    public static void main(String[] args)
    {
        // Esempio delle dispense: IterativeFib(4)
        int n = 4;
        reset();
        System.out.printf("iterazione | i        | next = prev + curr | prev | curr\n");
        long r = iterativeFib(n, true);
        System.out.printf("Il ciclo termina con i = %d > %d: restituisce curr = %d\n", n + 1, n, r);
        printSteps(2, 3, 5, 6, 7, 8, 9, 10, 11, 12, 14);
        System.out.printf("S(n) = 5n + 1 = %d\n", 5 * n + 1);

        // Confronto con Fib: passi di Fib = 4F(n+1) - 2 (formula verificata nella sezione precedente)
        System.out.printf("\n");
        System.out.printf("  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1\n");
        int[] values = {5, 10, 20, 30};
        for (int v : values)
        {
            reset();
            r = iterativeFib(v, false);
            long steps = total();
            System.out.printf("%3d | %6d | %16d | %12d | %d\n", v, r, 4 * iterativeFib(v + 1, false) - 2, steps, 5 * v + 1);
        }
    }
}

Writing IterativeFib.java


In [3]:
!javac IterativeFib.java && java IterativeFib

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
Il ciclo termina con i = 5 > 4: restituisce curr = 3
Riga | Esecuzioni
   2 |          1
   3 |          0
   5 |          1
   6 |          1
   7 |          1
   8 |          4
   9 |          3
  10 |          3
  11 |          3
  12 |          3
  14 |          1
Passi totali S = 21
S(n) = 5n + 1 = 21

  n |   F(n) | Fib: 4F(n+1) - 2 | IterativeFib | 5n + 1
  5 |      5 |               30 |           26 | 26
 10 |     55 |              354 |           51 | 51
 20 |   6765 |            43782 |          101 | 101
 30 | 832040 |          5385074 |          151 | 151


**Prova tu.** Cambia $n$ e controlla che i passi siano $5n + 1$.

In [4]:
%%writefile ProvaIterativeFib.java
public class ProvaIterativeFib
{
    public static void main(String[] args)
    {
        int n = 10;        // <-- cambia n
        IterativeFib.reset();
        System.out.printf("iterazione | i        | next = prev + curr | prev | curr\n");
        long r = IterativeFib.iterativeFib(n, true);
        System.out.printf("F(%d) = %d, S = %d = 5n + 1 = %d\n", n, r, IterativeFib.total(), 5 * n + 1);
    }
}

Writing ProvaIterativeFib.java


In [5]:
!javac ProvaIterativeFib.java && java ProvaIterativeFib

iterazione | i        | next = prev + curr | prev | curr
inizio     | 2        | -                  |    0 |    1
1          | 2 -> 3   | 0 + 1 = 1          |    1 |    1
2          | 3 -> 4   | 1 + 1 = 2          |    1 |    2
3          | 4 -> 5   | 1 + 2 = 3          |    2 |    3
4          | 5 -> 6   | 2 + 3 = 5          |    3 |    5
5          | 6 -> 7   | 3 + 5 = 8          |    5 |    8
6          | 7 -> 8   | 5 + 8 = 13         |    8 |   13
7          | 8 -> 9   | 8 + 13 = 21        |   13 |   21
8          | 9 -> 10  | 13 + 21 = 34       |   21 |   34
9          | 10 -> 11 | 21 + 34 = 55       |   34 |   55
F(10) = 55, S = 51 = 5n + 1 = 51


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).